# LanceDB fundamentals

In [1]:
import lancedb
from constants import KNOWLEDGE_BASE_PATH

vector_db = lancedb.connect(KNOWLEDGE_BASE_PATH)
vector_db

/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LanceDBConnection(/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/09_lancedb/knowledge_base)

In [2]:
import json

with open("animals_text_embeddings.json") as file:
    data = json.load(file)

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [3]:
vector_db.create_table("animals_text", exist_ok=True, data=data)
vector_db["animals_text"]

LanceTable(connection=LanceDBConnection(/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/09_lancedb/knowledge_base), name="animals_text")

In [5]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"


## Add more data

In [7]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

vector_db["animals_text"].add(more_data)

In [8]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"



## Create empty table

In [ ]:
from lancedb.pydantic import LanceModel


class Article(LanceModel):
    title: str
    content: str


vector_db.create_table("articles", schema=Article, exist_ok=True)

LanceTable(connection=LanceDBConnection(/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/09_lancedb/knowledge_base), name="articles")

In [11]:
vector_db.table_names()

['animals_text', 'articles']

TODO: add data to articles table

In [ ]:
articles_data = [
    {
        "title": "Fiskar",
        "content": """ 
Fiskar (Pisces) är en grupp vattenlevande ryggradsdjur med fenor, som indelas i benfiskar, broskfiskar och käklösa fiskar. De flesta arter andas med gälar och är växelvarma. Undantaget är lungfiskar. Eftersom landryggradsdjuren släktskapsmässigt är en typ av kvastfeniga fiskar men inte klassificeras som fiskar är begreppet "fiskar" parafyletiskt.

Vetenskapen om fiskar kallas iktyologi.
""",
    },
    {
        "title": "krabba",
        "content": """ 
Det finns stora variationer mellan de olika arternas storlek från släktet Pinnotheres som är bara några millimeter stor och den japanska spindelkrabban (Macrocheira kaempferi) som med utsträckta armar når en spann av 3,7 meter. 
Alla krabbor har ett tjockt exoskelett som huvudsakligen består av kalciumkarbonat. 
Liksom hos andra tiofotade kräftdjur finns fem ben på varje sida och det främre paret är utrustad med saxformiga klor (chela).
Hos simkrabbor (Portunidae) liknar det sista benparet paddlar som används för simning.[3]
""",
    },
]

vector_db["articles"].add(articles_data)

## Drop a table

In [16]:
vector_db.drop_table("articles")

In [17]:
vector_db.table_names()

['animals_text']

## Vector search in LanceDB

Flow here:
1. use embedding model to create vector embeddings for each document
2. use same embedding model to create vector embedding of the query
3. send in the query_vector to search for closest documents

If we want to do RAG -> send in closest document to LLM to use as context

In [18]:
vector_db["animals_text"].to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [19]:
query_vector = [0.5, 0.2, 0.9]

vector_db["animals_text"].search(query_vector).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]",0.2673
3,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]",0.2822
4,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]",0.4254
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]",0.5841
6,A small brown dog running.,"[0.12, 0.85, 0.33]",0.8918
7,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]",1.1541


## Embeddings API in LanceDB

- automatically generate embeddings

In [23]:
from lancedb.embeddings import get_registry

embedding_model = get_registry().get("cohere").create(name = "embed-multilingual-v3.0")

embedding_model.ndims()

1024

In [27]:
from lancedb.pydantic import Vector

class JokeModel(LanceModel):
    joke: str = embedding_model.SourceField()
    embedding: Vector(embedding_model.ndims()) = embedding_model.VectorField # type:ignore

vector_db.create_table("jokes", schema=JokeModel, exist_ok=True)
vector_db["jokes"]

LanceTable(connection=LanceDBConnection(/Users/kevinbruno/Documents/mlops/mlops-molnplattformar/llmops_kevin_bruno_mlo25/09_lancedb/knowledge_base), name="jokes")